# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a step-by-step guide for loading and exploring the FAIR² protein dataset using the [mlcroissant](https://github.com/mlcommons/croissant) library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

In [ ]:
# List all Record Sets with their `@id` fields
print("Available Record Sets:")
for record_set in metadata.record_sets:
    print(f"- Name: {record_set.name}, @id: {record_set.id}")
    print("  Fields:")
    for field in record_set.fields:
        print(f"    - Name: {field.name}, @id: {field.id}, type: {field.data_type}")
    print()

## 3. Data Extraction
Load data from one or more record sets into DataFrames for analysis. All record sets and fields are referenced by their `@id` as listed above.

In [ ]:
# Collect all Record Set `@id`s
record_set_ids = [rs.id for rs in metadata.record_sets]
dataframes = {}

# Load the records for each record set
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if len(records) > 0:
        df = pd.DataFrame(records)
        dataframes[record_set_id] = df
        print(f"Loaded {len(df)} records for record set '@id': {record_set_id}")
        print(f"Columns: {df.columns.tolist()}")
        print(df.head(), "\n")
    else:
        print(f"No records found for record set '@id': {record_set_id}\n")


## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and grouping data by attributes. 

In [ ]:
# ---
# Example: EDA for a protein-level record set
# Identify a record set containing protein quantification (look for 'abundance' or numeric fields).

# You should select these `@id`s based on the overview output above. You may adjust as needed.
protein_record_set_id = None
protein_numeric_field_id = None
protein_group_field_id = None

# Try to automatically pick likely numeric and categorical fields based on metadata.
for rs in metadata.record_sets:
    for field in rs.fields:
        if (field.data_type in ["Float", "Number", "Integer"] or 'abundance' in field.name.lower()):
            if protein_record_set_id is None:
                protein_record_set_id = rs.id
                protein_numeric_field_id = field.id
        if (field.data_type == "Text" and 'sample' in field.name.lower()):
            protein_group_field_id = field.id

if protein_record_set_id is None:
    # Fallback to first record set with data if automatic detection failed
    if len(dataframes) > 0:
        protein_record_set_id = list(dataframes.keys())[0]
        df = dataframes[protein_record_set_id]
        numeric_cols = df.select_dtypes(include='number').columns
        if len(numeric_cols) > 0:
            protein_numeric_field_id = numeric_cols[0]
        else:
            protein_numeric_field_id = df.columns[0]  # just as example

df = dataframes[protein_record_set_id]

print(f"Protein Record Set: {protein_record_set_id}")
print(f"Numeric Field for analysis: {protein_numeric_field_id}")
if protein_group_field_id:
    print(f"Group Field: {protein_group_field_id}")

# Remove rows with missing values in the numeric field
filtered_df = df.dropna(subset=[protein_numeric_field_id])

# Filter: Only keep values greater than an arbitrary threshold (e.g. >10)
threshold = 10
filtered_df = filtered_df[filtered_df[protein_numeric_field_id] > threshold]
print(f"Filtered records with {protein_numeric_field_id} > {threshold} (total={len(filtered_df)}):")
print(filtered_df[[protein_numeric_field_id]].head())

# Normalize the numeric field (Z-score)
filtered_df[f"{protein_numeric_field_id}_normalized"] = (
    filtered_df[protein_numeric_field_id] - filtered_df[protein_numeric_field_id].mean()
) / filtered_df[protein_numeric_field_id].std()
print(f"Normalized {protein_numeric_field_id} for filtered records:")
print(filtered_df[[protein_numeric_field_id, f"{protein_numeric_field_id}_normalized"]].head())

# Group by a categorical field (if available)
if protein_group_field_id and protein_group_field_id in filtered_df.columns:
    grouped_df = filtered_df.groupby(protein_group_field_id)[protein_numeric_field_id].mean().reset_index().sort_values(protein_numeric_field_id, ascending=False)
    print(f"Grouped data by {protein_group_field_id} (mean of {protein_numeric_field_id}):")
    print(grouped_df.head())

## 5. Visualization
Visualize data distributions or relationships between fields in the dataset.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Distribution of the selected numeric field
plt.figure(figsize=(8, 5))
sns.histplot(filtered_df[protein_numeric_field_id], bins=50, kde=True)
plt.title(f"Distribution of {protein_numeric_field_id}")
plt.xlabel(protein_numeric_field_id)
plt.ylabel("Count")
plt.tight_layout()
plt.show()

# If a group field is available, show group-wise boxplot
if protein_group_field_id and protein_group_field_id in filtered_df.columns:
    plt.figure(figsize=(10, 5))
    sns.boxplot(x=protein_group_field_id, y=protein_numeric_field_id, data=filtered_df)
    plt.title(f"{protein_numeric_field_id} by {protein_group_field_id}")
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.show()

## 6. Conclusion
In this notebook, we've demonstrated how to use the `mlcroissant` library to load metadata and records from a FAIR²-compliant Croissant schema. We reviewed available record sets and fields (referenced by their `@id`), extracted and processed data, and performed exploratory analyses including normalization and visualization of protein abundance or similar numeric fields. 

For further analysis, you can select specific fields or record sets relevant to your research question, always referencing the `@id`s for reproducibility and clarity.